# RPOE-X — Local Inference Test
Loads the trained LoRA adapter (`rpoe_x_grpo_adapter`) on top of `Qwen2.5-0.5B-Instruct`
and runs three routing/wheel-assignment scenarios to verify the model learned the correct policy.

In [ ]:
# Install deps if not already present
%pip install -q transformers peft accelerate torch

In [ ]:
import sys
import os
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT    = Path(os.path.abspath("../"))
ADAPTER_PATH = REPO_ROOT / "training" / "rpoe_x_grpo_adapter"
BASE_MODEL   = "Qwen/Qwen2.5-0.5B-Instruct"

# Add repo root so rpoe_x package imports work
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root    : {REPO_ROOT}")
print(f"Adapter path : {ADAPTER_PATH}")
print(f"Adapter exists: {ADAPTER_PATH.exists()}")
print(f"Base model   : {BASE_MODEL}")

In [ ]:
# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")

In [ ]:
# ── Load base model + LoRA adapter ───────────────────────────────────────────
print(f"Loading tokenizer from {BASE_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    device_map="auto",
)

print(f"Applying LoRA adapter from {ADAPTER_PATH} ...")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParams: {trainable:,} trainable / {total:,} total")
print("Model ready.")

In [ ]:
# ── Import prompt helpers from rpoe_x package ────────────────────────────────
from rpoe_x.training.train import (
    ORCH_SYSTEM,
    ZONE_SYSTEM,
    format_orch_obs,
    format_zone_obs,
    parse_action,
)

print("ORCH_SYSTEM:\n", ORCH_SYSTEM[:120], "...")
print("\nZONE_SYSTEM:\n", ZONE_SYSTEM[:120], "...")

In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────
def run_inference(system_prompt: str, user_content: str, max_new_tokens: int = 32) -> str:
    messages = [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": user_content},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # greedy — deterministic for tests
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return completion.strip()


def llm_orchestrator(obs: dict) -> dict:
    completion = run_inference(ORCH_SYSTEM, format_orch_obs(obs))
    action = parse_action(completion)
    if action is None or "zone_id" not in action:
        print(f"  [WARN] Unparseable completion: {completion!r}")
        return {"zone_id": -1, "raw": completion}
    action["raw"] = completion
    return action


def llm_zone_agent(obs: dict) -> dict:
    completion = run_inference(ZONE_SYSTEM, format_zone_obs(obs))
    action = parse_action(completion)
    if action is None or "wheel_id" not in action:
        print(f"  [WARN] Unparseable completion: {completion!r}")
        return {"wheel_id": -1, "raw": completion}
    action["raw"] = completion
    return action


print("Inference helpers defined: llm_orchestrator, llm_zone_agent")

In [ ]:
print("Inference Examples - Trained Agent Behavior\n")

# Example 1: High queue at zone 1
orch_obs_1 = {
    "zone_occupancy":    [0.45, 0.85, 0.60, 0.50, 0.40],
    "zone_queue_lengths": [2, 8, 1, 0, 1],  # Zone 1 is busy
    "zone_avg_wait":     [5.0, 25.0, 8.0, 3.0, 2.0],
    "arrival_rate_ema":  [0.01, 0.04, 0.015, 0.005, 0.008],
    "time_of_day": 0.5,
    "step": 150,
}

action_1 = llm_orchestrator(orch_obs_1)
print(f"Scenario 1: Zone 1 (Inorbit) has long queue (8 cars)")
print(f"  Input zones queue: {orch_obs_1['zone_queue_lengths']}")
print(f"  Agent decision: Route to zone {action_1['zone_id']}")
print(f"  Model raw output: {action_1.get('raw', '')}")
print(f"  Expected: Zone 1 (busiest) - {action_1['zone_id'] == 1}\n")

# Example 2: Wheel assignment
zone_obs_1 = {
    "zone_id": 1,
    "wheel_occupancy":    [0.95, 0.30, 0.88, 0.10],  # Wheel 3 is empty
    "wheel_queue_lengths": [2, 0, 1, 0],
    "est_rotation_cost":  [15.0, 2.0, 12.0, 1.0],
    "time_of_day": 0.5,
    "step": 150,
}

action_2 = llm_zone_agent(zone_obs_1)
print(f"Scenario 2: Wheel assignment at Inorbit")
print(f"  Input wheel occupancy: {[round(x, 2) for x in zone_obs_1['wheel_occupancy']]}")
print(f"  Agent decision: Assign to wheel {action_2['wheel_id']}")
print(f"  Model raw output: {action_2.get('raw', '')}")
print(f"  Expected: Wheel 3 (emptiest) - {action_2['wheel_id'] == 3}\n")

# Example 3: Peak hour congestion
orch_obs_2 = {
    "zone_occupancy":    [0.92, 0.88, 0.95, 0.85, 0.80],
    "zone_queue_lengths": [5, 3, 7, 2, 1],  # Zone 2 (Hitech Metro) busiest
    "zone_avg_wait":     [40.0, 30.0, 50.0, 25.0, 15.0],
    "arrival_rate_ema":  [0.035, 0.025, 0.045, 0.020, 0.015],
    "time_of_day": 0.75,  # Peak hour
    "step": 200,
}

action_3 = llm_orchestrator(orch_obs_2)
print(f"Scenario 3: Peak hour (time_of_day=0.75)")
print(f"  All zones high occupancy; zone 2 busiest with 7 cars waiting")
print(f"  Agent decision: Route to zone {action_3['zone_id']}")
print(f"  Model raw output: {action_3.get('raw', '')}")
print(f"  Expected: Zone 2 (queue=7) - {action_3['zone_id'] == 2}\n")

print("=" * 60)
passed = sum([
    action_1['zone_id'] == 1,
    action_2['wheel_id'] == 3,
    action_3['zone_id'] == 2,
])
print(f"Results: {passed}/3 scenarios matched expected output")
if passed == 3:
    print("Inference examples complete — model successfully learned routing & wheel policies.")
else:
    print("Some scenarios did not match — check raw outputs above for partial credit.")